In [ ]:
#Set up paths
import sys
import os

workdir_ = '/glade/work/juliob'
if ( workdir_ not in sys.path ):
    sys.path.append(workdir_)
    print( f" a path to {workdir_} added in {__name__} ")

# Cartopy for pretty maps
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature

from myPythonTools.Utils import utils as uti
from myPythonTools.Utils import numerical_utils as nuti
from myPythonTools.Utils import AveragingUtils as Av
from myPythonTools.Utils import validation_data as Val
from myPythonTools.Utils import PlotUtil as Pu
from myPythonTools.Plotting import LatLonMaps as LL 
from myPythonTools.CASutils import filter_utils as fu 

from PyRegridding.Utils import GridUtils as GrU
from PyRegridding.Utils import MakePressures as MkP
from PyRegridding.Drivers import RegridField as RgF

# The usual
from datetime import date
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.tri as tri

# Cartopy for pretty maps
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Some other useful packages 
import importlib
import copy
import time
import cftime
import yaml
import glob
#from box import Box #???

importlib.reload( LL )
importlib.reload( uti )
importlib.reload( nuti )
importlib.reload( Pu )
importlib.reload(Av)
importlib.reload(Val)
#importlib.reload(vAB)
importlib.reload(MkP)
importlib.reload(GrU)


##########################################################
# Read YAML file and open xarray Datasets
#---------------------------------------------------------

with open('configure_ana_plots.yaml', 'r') as file:
    cfg = yaml.safe_load(file)


x='ne240x2_QxQsst_smth'
x='ne240x2'

exp, subd, Src, Hkey, Dst, useri = cfg[x]['name'] , cfg[x]['subdir'] , cfg[x]['SrcGrid'] , cfg[x]['Hkey'] , cfg[x]['DstGrid'] , cfg[x]['user'] 
ymdPat = "2004-06-01_2004-06-30" #cfg[x]['ymdPat']
#ymdPat = "2004-07-01_2004-07-31" #cfg[x]['ymdPat']

Bdiro = f"/glade/derecho/scratch/juliob/archive/{exp}/atm/GWana/"

fname=f"{exp}_GWana_xPyP_{ymdPat}.nc"
print(Bdiro+fname )
XX = xr.open_mfdataset( Bdiro+fname )
fname=f"{exp}_GWana_zeta_{ymdPat}.nc"
print(Bdiro+fname )
Xzt = xr.open_mfdataset( Bdiro+fname )
fname=f"{exp}_GWana_PBLq_{ymdPat}.nc"
print(Bdiro+fname )
Xpbl = xr.open_mfdataset( Bdiro+fname )






clat2,clon2 = GrU.latlon( grid='ne16pg3' )
lat1,lon1 = GrU.latlon( grid='fv1x1' )

days = np.arange( len(XX.time.values) )/4.



In [ ]:
devUW_x1 = XX.uPomegaP.values
print( np.shape( devUW_x1 ) )
nt,nz,ny,nx = np.shape( devUW_x1 )


zeta_x1=Xzt.zeta.values
zeta2_x1=Xzt.zeta2.values


In [ ]:
#################################
# weird stuff for axes
from scipy.ndimage import gaussian_filter as GF
from matplotlib.ticker import FuncFormatter

def plot_with_formatter(ax, xtick_positions_m, m_per_deg, ytick_positions_s):

    def m_to_lon(x):
        return (x / m_per_deg) #- 180.

    def longitude_formatter(x, pos):
        degree_symbol = '\u00B0'
        lon_fix = m_to_lon(x)
        if ( lon_fix == 180. ):
            hemi_symbol=''
        elif ( lon_fix < 180. ):
            hemi_symbol='E'
        elif ( lon_fix > 180. ):
            lon_fix = np.abs(lon_fix-360.)
            hemi_symbol='W'
            
        #return f'{m_to_lon(x):.1f}{degree_symbol}'
        return f'{lon_fix:.1f}{degree_symbol}{hemi_symbol}'

    def s_to_day(y):
        return (y / 86400. )+1. #- 180.

    def day_formatter(y, pos):
        return f'{s_to_day(y):.1f}'

    ax.set_xticks(xtick_positions_m)
    ax.xaxis.set_major_formatter(FuncFormatter(longitude_formatter))
    ax.set_yticks(ytick_positions_s)
    ax.yaxis.set_major_formatter(FuncFormatter(day_formatter))


inset_font_size=18
ax_lab_size = 16
fldoo=r"$ \overline{ \rho u'w'}$  "
fldoo=r"$ \overline{u'\omega'}$  "


lat_sel= uti.find_nearest_plev_indices(plev=lat1  , target_levels=[-41.])
#lat_sel= uti.find_nearest_plev_indices(plev=lat1  , target_levels=[-49.])

jj = lat_sel[0] # 45 #45 #40


latval = lat1[jj]

circum = 2.*np.pi*6_371_000* np.cos( np.radians( latval ) ) # circumfrerence along lat in m
if latval < 0:
    ILat=f'{str(int(-latval))}S'
else:
    ILat=f'{str(int(latval))}N'

m_per_deg = circum / 360. 

# Define tick positions in degrees and convert to km
#tick_positions_deg = [60, 120, 180, 240, 300]
xtick_positions_deg = [-120, -60, 0, 60, 120]
xtick_positions_m = [(deg+180.)* m_per_deg for deg in xtick_positions_deg]  # Assuming 68 km per degree
ytick_positions_day =[1,10,20,30]
ytick_positions_s = [(day-1) * 86400.  for day in ytick_positions_day]

##########################################################
# Make fancy plots
#---------------------------------------------------------

fig, ax = plt.subplots(1, 1, figsize=(10, 8))

# Flatten axs to easily access each subplot by index
#ax = ax.flatten()
ax = np.atleast_1d(ax)

dolev = np.linspace(-1,1,num=41)
olev = np.linspace(-10,10,num=21)
vorlev=1.* np.linspace( 0, 0.0005, num=31 )

tootrange = f" {(XX.time[0].values.item() ).strftime('%Y-%m-%d')} to {(XX.time[-1].values.item() ).strftime('%Y-%m-%d')}"
dayrange=f"{(XX.time[0].values.item() ).strftime('%Y-%m-%d')}-{(XX.time[-1].values.item() ).strftime('%d')}"

xx=np.linspace(0,1.,num=nx)*circum     #( 2.*np.pi*6_371_000* np.cos( np.radians( latval ) ))
tt=np.linspace(0,1.,num=nt)*6.*3600.*nt 


print(f" Lat={latval:.1f} ")
print(tootrange )

plev=XX.lev.values
zlev=-7.0*np.log(plev/1_000.)
klist=np.arange( len(zlev) )
p=0
k=4
k_vort=2 #1
print( plev[k] , zlev[k] )
scak = 500./ plev[k]
if zlev[k]>30. :
    scak=scak*0.1


co=ax[p].contourf(xx, tt, GF( np.average( devUW_x1[:,k,jj-5:jj+5,:] ,axis=1 ), sigma=[1,2] ),levels=1*dolev/scak ,cmap='bwr' , extend='both' )

col=ax[p].contour(xx, tt, GF( np.sqrt( np.average( zeta2_x1[:,k_vort,jj-5:jj+5,:] ,axis=1 ) ),sigma=[1,2]),levels=vorlev[[4,6,7,30]] , colors='black', alpha=.5) #, colors='black' , alpha=.2)

ln0=ax[p].plot( tt*5 , tt, color='black'  ,  label='5ms$^{-1}$'   )
ln0=ax[p].plot( tt*10 , tt, '--', color='black' ,  label='10ms$^{-1}$' )
ln0=ax[p].plot( tt*20 , tt, '-.', color='black' ,  label='20ms$^{-1}$' )

#ax[p].set_title( f'Overlay' )
ax[p].set_ylabel( 'Days' , fontsize=ax_lab_size )
ax[p].set_xlabel( 'Longitude' , fontsize=ax_lab_size )
ax[p].set_xlim( 0,xx.max())

print( np.shape(xx))
print( nx/2 , xx[0], xx[100] )

ax[p].text(
    x=xx[int(4.8*nx)//6], y=2*86400.,              # Coordinates in data space
    s='Andes',        # Text string
    fontsize=inset_font_size,
    ha='center', va='center', 
    bbox=dict(facecolor='white', alpha=0.8, edgecolor='black')
)
ax[p].text(
    x=xx[int(0.4*nx)], y=1*86400.,              # Coordinates in data space
    s='Aus',        # Text string
    fontsize=inset_font_size,
    ha='center', va='center', 
    bbox=dict(facecolor='white', alpha=0.8, edgecolor='black')
)
ax[p].text(
    x=xx[int(0.47*nx)], y=3*86400.,              # Coordinates in data space
    s='NZ',        # Text string
    fontsize=inset_font_size,
    ha='center', va='center', 
    bbox=dict(facecolor='white', alpha=0.8, edgecolor='black')
)




var1x = GF( np.average( devUW_x1[:,k,jj-5:jj+5,:] ,axis=1 ) ,sigma=[1,2] )
var2x = GF( np.sqrt( np.average( zeta2_x1[:,k_vort,jj-5:jj+5,:] ,axis=1 ) ),sigma=[1,2])


power1= 1. #0.25 #0.50 #1.0 # 0.0625
power2= 1. #0.25 #0.50 # 1.0 # 0.0625

coco=np.zeros( nx )

aa=np.corrcoef( np.abs(var1x[:,0:100].flatten())**power1 , np.abs(var2x[:,0:100].flatten())**power2 ) 
print( aa[0,1] , XX.lon.values[[0,100]] )
corrE = aa[0,1]
aa=np.corrcoef( np.abs(var1x[:,150:210].flatten())**power1 , np.abs(var2x[:,150:210].flatten())**power2 ) 
print( aa[0,1] , XX.lon.values[[150,210]] -360. )
corrC = aa[0,1]
aa=np.corrcoef( np.abs(var1x[:,250:-1].flatten())**power1 , np.abs(var2x[:,250:-1].flatten())**power2 ) 
print( aa[0,1], XX.lon.values[[250,-1]] -360. )
corrW = aa[0,1]

for x in np.arange( nx ):
    aa=np.corrcoef( np.abs(var1x[:,x])**power1 , np.abs(var2x[:,x])**power2 ) 
    coco[x] = aa[0,1]



ax[p].text(
    x=xx[int(0.15*nx)], y=22*86400.,              # Coordinates in data space
    s=f'E basin \n$r$={corrE:.2f}',        # Text string
    fontsize=inset_font_size,
    ha='center', va='center', 
    bbox=dict(facecolor='white', alpha=1.0, edgecolor='black')
)
ax[p].text(
    x=xx[int(0.65*nx)], y=15*86400.,              # Coordinates in data space
    s=f'C basin \n$r$={corrC:.2f}',        # Text string
    fontsize=inset_font_size,
    ha='center', va='center', 
    bbox=dict(facecolor='white', alpha=1.0, edgecolor='black')
)
ax[p].text(
    x=xx[int(0.95*nx)], y=10*86400.,              # Coordinates in data space
    s=f'W basin \n$r$={corrW:.2f}',        # Text string
    fontsize=inset_font_size,
    ha='center', va='center', 
    bbox=dict(facecolor='white', alpha=1.0, edgecolor='black')
)

# Set the tick positions and formatter for the x-axis
#ax[p].set_xticks(tick_positions_m)
#ax[p].xaxis.set_major_formatter(FuncFormatter(longitude_formatter(tick_positions_m,m_per_deg,))
plot_with_formatter(ax[p], xtick_positions_m, m_per_deg,ytick_positions_s)
ax[p].legend(fontsize=inset_font_size,loc="upper right")
ax[p].text(-0.08, 1.05, f"{chr(97 + p+1 )})", transform=ax[p].transAxes,
    fontsize=16, fontweight='bold', va='top')
ax[p].tick_params(labelsize=ax_lab_size, length=5, width=1) 

cb=plt.colorbar( co )
cb.set_ticks([-0.1, -0.05, 0, 0.05, 0.1])     
cb.set_ticklabels(["−0.1", "−0.05", "0", "0.05", "0.1"])


cb.ax.tick_params(labelsize=16, length=5, width=1)  # size/length/width
cb.set_label(f"{fldoo}", fontsize=16, labelpad=8)

#fig.suptitle( f'{exp}, {tootrange}, Lat=<{latval-5:.0f},{latval+5:.0f}>' , fontsize=14) #20)

FigName=f'Fig_Hpass-devUW-xtHovmoeller_y{ILat}_{dayrange}_{exp}.png'

print(FigName)
Pu.save_link(png_filename = FigName , dpi=600 )